In [1]:
# Cell 0: Configure GPT variant, paths, and run toggles.
import os
import re
import ast
import json
import textwrap
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
ENV_FILE = PROJECT_ROOT / 'env.txt'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'

# GPT family variant (open-source GPT-2 checkpoints for LoRA experimentation).
GPT_VARIANT = 'gpt2_large'  # choose: gpt2, gpt2_medium, gpt2_large, gpt2_xl
GPU_ID = None               # set to e.g. '0' to pin this notebook to one GPU

RUN_TRAIN = True
RUN_INFERENCE = True
RUN_EVAL = True

TRAIN_ROWS = 12000   # use None for full express.csv
MAX_SOURCE_LEN = 384
MAX_NEW_TOKENS = 48
TRAIN_BATCH_SIZE = 4
INFER_BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 2e-4
EPOCHS = 2.0

# Best-checkpoint and early stopping controls.
LOAD_BEST_MODEL_AT_END = True
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
EVAL_SPLIT_RATIO = 0.1
MIN_EVAL_SAMPLES = 200

if GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'


In [2]:
# Cell 1: Load token, helper functions, and selected GPT variant spec.
import pickle
import yaml
import pandas as pd


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if not env_path.exists():
        raise FileNotFoundError(f'Env file not found: {env_path}')

    token = ''
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

    if not token:
        raise ValueError(f'HF_TOKEN not found or empty in {env_path} or {pkl_path}')
    return token


def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(str(x) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-100:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Loaded HF token using', ENV_PKL_FILE, 'with fallback to', ENV_FILE)

GPT_MAP = {
    'gpt2': {
        'model_id': 'gpt2',
        'repo_id': 'gpt2',
        'size_b': 0.124,
        'baseline_file': 'gpt_35_turbo_0125.csv',
        'label': 'GPT-2 + LoRA',
    },
    'gpt2_medium': {
        'model_id': 'gpt2_medium',
        'repo_id': 'gpt2-medium',
        'size_b': 0.355,
        'baseline_file': 'gpt_35_turbo_0125.csv',
        'label': 'GPT-2-medium + LoRA',
    },
    'gpt2_large': {
        'model_id': 'gpt2_large',
        'repo_id': 'gpt2-large',
        'size_b': 0.774,
        'baseline_file': 'gpt_35_turbo_0125.csv',
        'label': 'GPT-2-large + LoRA',
    },
    'gpt2_xl': {
        'model_id': 'gpt2_xl',
        'repo_id': 'gpt2-xl',
        'size_b': 1.5,
        'baseline_file': 'gpt_35_turbo_0125.csv',
        'label': 'GPT-2-xl + LoRA',
    },
}

if GPT_VARIANT not in GPT_MAP:
    raise ValueError(f'GPT_VARIANT must be one of {list(GPT_MAP.keys())}')

SPEC = GPT_MAP[GPT_VARIANT]
MODEL_ID = SPEC['model_id']
SCRATCH_MODEL_ROOT = Path('/scratch/rameyjm7/models')
LOCAL_MODEL_DIR = SCRATCH_MODEL_ROOT / SPEC['repo_id'].replace('-', '_')
MODEL_NAME_OR_PATH = str(LOCAL_MODEL_DIR) if LOCAL_MODEL_DIR.exists() else SPEC['repo_id']
MODEL_SIZE_B = float(SPEC['size_b'])

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_DIR = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_express_lora'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / MODEL_ID
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / MODEL_ID

TRAIN_PREP_PATH = METRICS_DIR / f'{MODEL_ID}_train_prep_stats.json'
EVAL_INPUT_PATH = METRICS_DIR / f'{MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_clean.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_lora_overlay_{MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_DIR, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('GPT_VARIANT =', GPT_VARIANT)
print('MODEL_ID =', MODEL_ID)
print('MODEL_NAME_OR_PATH =', MODEL_NAME_OR_PATH)
print('Expected local model dir =', LOCAL_MODEL_DIR)
print('Using local model dir =', LOCAL_MODEL_DIR.exists())
print('RUN_DIR =', RUN_DIR)
print('METRICS_DIR =', METRICS_DIR)


Loaded HF token using /home/rameyjm7/workspace/llamafactory-gemma-lora/configs/env.pkl with fallback to /home/rameyjm7/workspace/llamafactory-gemma-lora/env.txt
GPT_VARIANT = gpt2_large
MODEL_ID = gpt2_large
MODEL_NAME_OR_PATH = gpt2-large
Expected local model dir = /scratch/rameyjm7/models/gpt2_large
Using local model dir = False
RUN_DIR = /home/rameyjm7/workspace/llamafactory-gemma-lora/outputs/lora_runs/gpt2_large_express_lora
METRICS_DIR = /home/rameyjm7/workspace/llamafactory-gemma-lora/outputs/metrics/gpt2_large


In [3]:
# Cell 2: Build train/eval dataframes for the selected GPT variant.
train_source = EXPRESS_ROOT / 'data' / 'express.csv'
eval_source = EXPRESS_ROOT / 'results' / SPEC['baseline_file']

train_df = pd.read_csv(train_source)
if TRAIN_ROWS is not None:
    train_df = train_df.head(TRAIN_ROWS)
train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

eval_df = pd.read_csv(eval_source)
eval_df = eval_df[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(EVAL_INPUT_PATH, index=False)

print('Train rows:', len(train_df), 'from', train_source)
print('Eval rows:', len(eval_df), 'from', eval_source)
print('Saved eval input:', EVAL_INPUT_PATH)

display(train_df.head(2))
display(eval_df.head(2))


Train rows: 12000 from /home/rameyjm7/workspace/llamafactory-gemma-lora/express-emotion-recognition/data/express.csv
Eval rows: 35176 from /home/rameyjm7/workspace/llamafactory-gemma-lora/express-emotion-recognition/results/gpt_35_turbo_0125.csv
Saved eval input: /home/rameyjm7/workspace/llamafactory-gemma-lora/outputs/metrics/gpt2_large/gpt2_large_baseline_eval_input.csv


,segment,labels,number_of_labels
0,I feel <mask> by others admiration. If other a...,"['disgusted', 'disgusted']",2
1,"Question, is there a resource for novices that...",['confident'],1


,segment,labels,number_of_labels
0,I feel <mask> by others admiration. If other a...,"['disgusted', 'disgusted']",2
1,"Question, is there a resource for novices that...",['confident'],1


In [4]:
from datetime import datetime, timezone
# Cell 3: Train GPT LoRA (causal LM) for the selected variant.
if RUN_TRAIN:
    import torch
    from torch.utils.data import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback
    from peft import LoraConfig, TaskType, get_peft_model

    def parse_labels(raw):
        if isinstance(raw, list):
            return [str(x).strip().lower() for x in raw]
        if not isinstance(raw, str):
            return None
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x).strip().lower() for x in val]
        except Exception:
            return None
        return None

    def make_prompt(segment, num_labels):
        return textwrap.dedent(f"""
        You are an assistant that predicts masked emotion words in a Reddit post.
        Predict exactly {num_labels} masked token(s).
        Return ONLY a Python list of lowercase emotion words with length {num_labels}.

        Text:
        {segment}

        Answer:
        """).strip()

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    samples = []
    skipped = 0
    for row in train_df.itertuples(index=False):
        labels = parse_labels(row.labels)
        if labels is None:
            skipped += 1
            continue
        prompt = make_prompt(str(row.segment), int(row.number_of_labels))
        target = str(labels)
        samples.append({'prompt': prompt, 'target': target})

    TRAIN_PREP_PATH.write_text(
        json.dumps({'total_rows': int(len(train_df)), 'kept_rows': int(len(samples)), 'skipped_rows': int(skipped)}, indent=2),
        encoding='utf-8',
    )
    print('Train prep:', {'total_rows': len(train_df), 'kept_rows': len(samples), 'skipped_rows': skipped})

    class CausalListDataset(Dataset):
        def __init__(self, rows):
            self.rows = rows
        def __len__(self):
            return len(self.rows)
        def __getitem__(self, idx):
            return self.rows[idx]

    eval_dataset = None
    if len(samples) >= max(MIN_EVAL_SAMPLES, 2):
        eval_count = max(1, int(len(samples) * EVAL_SPLIT_RATIO))
        eval_rows = samples[-eval_count:]
        train_rows = samples[:-eval_count]
        if len(train_rows) >= 1 and len(eval_rows) >= 1:
            train_dataset = CausalListDataset(train_rows)
            eval_dataset = CausalListDataset(eval_rows)
        else:
            train_dataset = CausalListDataset(samples)
    else:
        train_dataset = CausalListDataset(samples)

    def collate_fn(features):
        prompts = [f['prompt'] for f in features]
        targets = [f['target'] for f in features]
        full_texts = [p + ' ' + t for p, t in zip(prompts, targets)]

        batch = tokenizer(
            full_texts,
            max_length=MAX_SOURCE_LEN,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )

        prompt_batch = tokenizer(
            prompts,
            max_length=MAX_SOURCE_LEN,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )

        labels = batch['input_ids'].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        for i in range(labels.shape[0]):
            prompt_len = int((prompt_batch['attention_mask'][i] == 1).sum().item())
            labels[i, :prompt_len] = -100

        batch['labels'] = labels
        return batch

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        target_modules=['c_attn', 'c_proj'],
    )

    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()

    use_cuda = torch.cuda.is_available()
    use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
    use_fp16 = bool(use_cuda and not use_bf16)

    eval_enabled = eval_dataset is not None
    trainer_args = TrainingArguments(
        output_dir=str(RUN_DIR / 'trainer_state'),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        logging_steps=20,
        save_strategy='epoch',
        eval_strategy='epoch' if eval_enabled else 'no',
        load_best_model_at_end=bool(LOAD_BEST_MODEL_AT_END and eval_enabled),
        metric_for_best_model='eval_loss' if eval_enabled else None,
        greater_is_better=False if eval_enabled else None,
        remove_unused_columns=False,
        fp16=use_fp16,
        bf16=use_bf16,
        report_to=[],
    )

    callbacks = []
    if eval_enabled and LOAD_BEST_MODEL_AT_END:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD))

    trainer = Trainer(
        model=model,
        args=trainer_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
        callbacks=callbacks,
    )

    train_out = trainer.train()
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(RUN_DIR))
    tokenizer.save_pretrained(str(RUN_DIR))

    summary = {
        'train_runtime': float(getattr(train_out, 'metrics', {}).get('train_runtime', 0.0)),
        'train_samples': int(len(train_dataset)),
        'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
        'model_id': MODEL_ID,
        'model_name_or_path': MODEL_NAME_OR_PATH,
    }
    (RUN_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('Saved LoRA adapter to:', RUN_DIR)
else:
    print('Training skipped (RUN_TRAIN=False).')


I0000 00:00:1775072117.463601 1914103 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Train prep: {'total_rows': 12000, 'kept_rows': 12000, 'skipped_rows': 0}


model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# Cell 4: Run GPT LoRA inference for selected variant and save raw outputs.
if RUN_INFERENCE:
    import torch
    from tqdm import tqdm
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    eval_df = pd.read_csv(EVAL_INPUT_PATH)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(RUN_DIR))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()

    def make_prompt(segment, num_labels):
        return textwrap.dedent(f"""
        You are an assistant that predicts masked emotion words in a Reddit post.
        Predict exactly {num_labels} masked token(s).
        Return ONLY a Python list of lowercase emotion words with length {num_labels}.

        Text:
        {segment}

        Answer:
        """).strip()

    def parse_pred_list(raw_text, expected_n):
        txt = str(raw_text).strip()
        m = re.search(r'\[[^\]]*\]', txt)
        if not m:
            return 'INVALID OUTPUT'
        candidate = m.group(0)
        try:
            val = ast.literal_eval(candidate)
        except Exception:
            return 'INVALID OUTPUT'
        if not isinstance(val, list):
            return 'INVALID OUTPUT'
        parsed = [str(x).strip().lower() for x in val]
        if len(parsed) != int(expected_n):
            return 'INVALID OUTPUT'
        return str(parsed)

    outputs = [None] * len(eval_df)

    for start in tqdm(range(0, len(eval_df), INFER_BATCH_SIZE), desc=f'{MODEL_ID} inference'):
        chunk = eval_df.iloc[start:start + INFER_BATCH_SIZE]
        prompts = [make_prompt(str(r.segment), int(r.number_of_labels)) for r in chunk.itertuples(index=False)]

        enc = tokenizer(
            prompts,
            max_length=MAX_SOURCE_LEN,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        input_ids = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)

        with torch.inference_mode():
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        decoded = []
        for i in range(gen.shape[0]):
            prompt_len = int(attention_mask[i].sum().item())
            new_ids = gen[i, prompt_len:]
            decoded.append(tokenizer.decode(new_ids, skip_special_tokens=True).strip())

        for i, txt in enumerate(decoded):
            n = int(chunk.iloc[i]['number_of_labels'])
            outputs[start + i] = parse_pred_list(txt, n)

    raw_df = eval_df.copy()
    raw_df['output'] = outputs
    raw_df.to_csv(RAW_OUTPUT_PATH, index=False)

    print('Saved raw outputs:', RAW_OUTPUT_PATH)
    print('Rows:', len(raw_df))
else:
    print('Inference skipped (RUN_INFERENCE=False).')


In [ ]:
# Cell 5: Clean and evaluate with existing scripts.
lora_metrics = {}
if RUN_EVAL:
    _ = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(RAW_OUTPUT_PATH),
            '--output', str(CLEAN_OUTPUT_PATH),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    eval_stdout = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(CLEAN_OUTPUT_PATH),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('LoRA metrics:', lora_metrics)
else:
    print('Evaluation skipped (RUN_EVAL=False).')


In [ ]:
from datetime import datetime, timezone
# Cell 6: Upsert this GPT variant into shared metrics table.
row = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'model_id': MODEL_ID,
    'family': 'gpt',
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': 'hf_peft_causal',
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)

print('Updated metrics table:', METRICS_TABLE_PATH)
display(metrics_df.tail(12))


In [ ]:
# Cell 7: Plot Figure 2 baseline + completed GPT LoRA variants.
import seaborn as sns
import matplotlib.pyplot as plt

lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p)})
fig_df = pd.DataFrame(rows)

gpt_lora_rows = []
for variant, spec in GPT_MAP.items():
    clean_path = PROJECT_ROOT / 'outputs' / 'metrics' / spec['model_id'] / f"{spec['model_id']}_lora_clean.csv"
    if clean_path.exists():
        gpt_lora_rows.append({
            'variant': variant,
            'model_id': spec['model_id'],
            'label': spec['label'],
            'size_b': float(spec['size_b']),
            'accv': compute_accv(clean_path),
        })

gpt_lora_df = pd.DataFrame(gpt_lora_rows)

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

if not gpt_lora_df.empty:
    gpt_lora_df = gpt_lora_df.sort_values('size_b')
    plt.scatter(
        gpt_lora_df['size_b'], gpt_lora_df['accv'],
        color=family_colors['GPT'], marker='*', s=260, edgecolors='black', linewidths=0.9,
        label='GPT + LoRA'
    )
    for r in gpt_lora_df.itertuples(index=False):
        plt.text(r.size_b * 1.04, r.accv + 0.004, r.label, fontsize=9)

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
y_max = 0.40
if not gpt_lora_df.empty:
    y_max = max(y_max, float(gpt_lora_df['accv'].max()) + 0.03)
plt.ylim(0.08, y_max)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title('Figure 2 Baseline + GPT LoRA Variants')
plt.legend(loc='lower left')
plt.tight_layout()

plt.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', FIGURE_PATH)
print('Completed GPT LoRA variants:', gpt_lora_df['model_id'].tolist() if not gpt_lora_df.empty else [])
